# Analyse calibrations from the prototype: `run_multiple_locations.py`

In [1]:
import os
import gc
import importlib
from copy import deepcopy
from dataclasses import dataclass, field
from pathlib import Path
from typing import Union

import epiweeks
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from rich.jupyter import display

from inframind_proteus.empirical_data import DiseaseTimeSeriesVariables
from inframind_proteus.outbreak_dynamics import RenewalSimulator, build_calibration_params_df, build_initial_infec_df
from inframind_proteus.empirical_data import DiseaseTimeSeriesVariables, DiseaseTimeSeriesCache
from inframind_proteus.outbreak_dynamics.utils import (
    load_yaml_dict, apply_include_exclude_logic, map_parallel_or_sequential, save_yaml_dict
)

In [2]:
@dataclass
class LocationSeasonSavedResults:
    """Prototype representation of results from `run_multiple_locations.py`.
    """

    main_out_dir: Path
    location_id: Union[str, int]
    year: int
    vars: DiseaseTimeSeriesVariables = field(
        default_factory=lambda: DiseaseTimeSeriesVariables(),
    )

    @classmethod
    def from_dir(cls, main_out_dir: Union[Path, str], location_id: Union[str, int], year: int):
        out_dir = Path(main_out_dir)

        return cls(
            main_out_dir=main_out_dir,
            location_id=location_id,
            year=year,
        )


    def load(self):
        main_out_dir = self.main_out_dir
        location_id = self.location_id
        year = self.year

        out_dir = main_out_dir / "calibration_results" / f"{location_id}_{year}"

        #==== Loaded data
        config_dict = load_yaml_dict(out_dir / "config.yaml")
        self.config = RenewalSimulator.from_config_dict(config_dict).config
        case_beam_selected_df = df =  pd.read_csv(out_dir / "case_beam_selected_df.csv.gz", index_col=["i_simulation", "quantile"])
        df.columns = pd.to_datetime(df.columns)
        self.params_df = pd.read_csv(out_dir / "params.csv.gz", index_col="i_simulation")
        self.scoring_summary_df: pd.DataFrame = pd.read_csv(out_dir / "scoring.csv.gz", index_col="i_simulation")
        self.observations_sr: pd.Series = DiseaseTimeSeriesCache(Path("../../data/disease/dengue_cases_uf_weekly")).get_location(location_id)


        # === Processed data
        self.selected_i_simulations:pd.DataFrame = case_beam_selected_df.index.get_level_values("i_simulation").unique()
        self.explored_param_names = list(self.config.sampling.param_ranges.keys())





In [3]:
# Parameters / Configs for this code
# ==================================

main_out_dir = Path("../../outputs/prototype_run_dynamic_model")
use_location_ids = ["SP", "PI", "GO", "RS"]
exclude_location_ids = ["CE"]  # Just to test.
use_years = list(range(2016, 2025))

uf_table_df = pd.read_csv(Path("../../data/demographic/uf_table.csv"))

# =========

# Combine all years and locations to be run
location_ids = apply_include_exclude_logic(
    uf_table_df["uf"],
    include_list=use_location_ids,
    exclude_list=exclude_location_ids,
)
years = list(use_years)
location_year_index = pd.MultiIndex.from_product(
    [location_ids, years],
    names=["uf", "season"]
)


# =======


In [7]:
# Do all the processing without keeping results in memory
# =================

# keys = list()
records = list()

def collect_calibration_records(location_year_tuple):
    location_id, year = location_year_tuple
    print(f"Iterating for {location_id} in {year}...")

    results = LocationSeasonSavedResults.from_dir(
        main_out_dir=main_out_dir,
        location_id=location_id,
        year=year,
    )

    try:
        results.load()
    except FileNotFoundError as e:
        print(f"Error loading results: {results.main_out_dir}, {results.location_id}, {results.year=}. Skipped.")
        return None

    # ======
    record = dict()

    selected_scores = results.scoring_summary_df.loc[results.selected_i_simulations]
    selected_params_df = results.params_df.loc[results.selected_i_simulations]

    record["location_id"] = location_id
    record["year"] = year

    for param_name in results.explored_param_names:
        record[f"{param_name}_mean"] = selected_params_df[param_name].mean()
        record[f"{param_name}_q02.5%"] = selected_params_df[param_name].quantile(0.025)
        record[f"{param_name}_q97.5%"] = selected_params_df[param_name].quantile(0.975)
    # record["rt_logist_r_high_mean"] = selected_params_df["rt_logist_r_high"].mean()
    # record["rt_logist_r_high_q02.5%"] = selected_params_df["rt_logist_r_high"].quantile(0.025)
    # record["rt_logist_r_high_q97.5%"] = selected_params_df["rt_logist_r_high"].quantile(0.975)

    return record

# Load one example to have general variables
sample_results = LocationSeasonSavedResults.from_dir(
        main_out_dir=main_out_dir,
        location_id="SP",
        year=2023,
    )
sample_results.load()


# Load all resulds and keep summaries in the records list
records = map_parallel_or_sequential(
    collect_calibration_records,
    location_year_index,
    ncpus=8
)


df = calibration_summary_df = pd.DataFrame.from_records(records)

Iterating for SP in 2016...Iterating for SP in 2018...Iterating for SP in 2022...
Iterating for SP in 2020...


Iterating for PI in 2017...Iterating for PI in 2019...
Iterating for SP in 2024...Iterating for PI in 2021...


Iterating for PI in 2016...
Iterating for PI in 2020...
Iterating for SP in 2023...
Iterating for PI in 2022...
Iterating for SP in 2017...
Iterating for SP in 2019...
Iterating for PI in 2018...
Iterating for SP in 2021...
Iterating for PI in 2023...
Iterating for GO in 2016...
Iterating for GO in 2018...
Iterating for GO in 2020...
Iterating for GO in 2022...
Iterating for GO in 2024...
Iterating for RS in 2017...
Iterating for RS in 2019...
Iterating for PI in 2024...
Iterating for GO in 2017...
Iterating for GO in 2019...
Iterating for GO in 2021...
Iterating for RS in 2018...
Iterating for RS in 2021...
Iterating for GO in 2023...
Iterating for RS in 2016...
Iterating for RS in 2020...
Iterating for RS in 2023...
Iterating for RS in 2022...
Iterating for RS in 

In [14]:
df = calibration_summary_df

# Reproduction number over the years
# ======
fig = px.line(
    df,
    x="year",
    y="rt_logist_r_high_mean",
    color="location_id",
    # error_y=df["rt_logist_r_high_q97.5%"] - df["rt_logist_r_high_mean"],
    # error_y_minus=df["rt_logist_r_high_mean"] - df["rt_logist_r_high_q02.5%"],
    markers=True
)

display(fig)


# Correlation betwen parameters
# ==============
x_param = "rt_logist_r_high"
y_param = "notif_relative_scale"

df = calibration_summary_df
param_names = sample_results.explored_param_names
if x_param in param_names and y_param in param_names:

    fig = px.scatter(
        df,
        x=f"{x_param}_mean",
        y=f"{y_param}_mean",
        color="location_id",
        hover_data=["year"],
    )

    display(fig)
    fig.write_html("../../.local/calibration_param_scatter.html")

else:
    print(f"One of the parameters {x_param} or {y_param} not found")

In [11]:
param_names

['notif_relative_scale',
 'rt_logist_dt_center',
 'rt_logist_dt_end',
 'rt_logist_r_high',
 'rt_logist_start',
 'rt_logist_w_center']